In [1]:
import os
import platform

In [4]:
platform.platform()[0]

'W'

In [5]:
class Binary():
    """
    Binary Class
    """
    def decimal_to_binary(self, decimal_num):
        """
        This function converts base 10 to base 2
        """
        binary_num = ""
        while decimal_num > 0:
            binary_num = str(decimal_num % 2) + binary_num
            decimal_num = decimal_num // 2
        # Fill to 16 digits with 0
        if len(binary_num) < 16:
            binary_num = "0"*(16-len(binary_num)) + binary_num
        return binary_num
        
    def binary_to_decimal(self, binary_num):
        """
        This function converts base 2 to base 10
        """
        decimal_num = 0
        for i in range(len(binary_num)):
            decimal_num += int(binary_num[i]) * (2 ** (len(binary_num)-i-1))
        return decimal_num
    
    def binary_crop(self, digit, binary_num):
        """
        This function crops the last n digits of the binary number
        """
        return binary_num[len(binary_num)-digit:]

    def binary_twos_complement(self, number):
        """
        This functions converts the (negative) number to its 16-bit two's complement representation
        """
        if number < 0:
            number = (1 << 16) + number  # Adding 2^16 to the negative number
        return number
    
    def binary_reverse_twos_complement(self, number):
        """
        This functions converts the 16-bit two's complement number back to its original signed representation 
        """
        if number & (1 << 15):  # Check if the most significant bit is 1
            number = number - (1 << 16)  # Subtract 2^16 from the number
        return number

In [27]:
binary = Binary()
a = binary.binary_twos_complement(-3600)
print(a)  # Example usage

print(binary.binary_reverse_twos_complement(0x0010))  # Example usage

61936
16


# Connect Test

In [1]:
import json
from pymodbus.client import ModbusSerialClient
import time

msg = {
    "mode": "Connect",
    "action": "connect_port",
    "port": "COM3"
}

In [2]:
com_port = msg.get("port")
client = ModbusSerialClient(
    port=com_port,
    baudrate=19200,
    parity="E",     
    stopbits=1,
    bytesize=8,
    timeout=1
)

if client.connect():
    print("Connected successfully")
    msg = f"Connected to COM{com_port}!"
    sta = "success"
    client.close()
else:
    print("Connection failed")
    msg = f"Can't connect to COM{com_port}"
    sta = "failed"
    client.close()

response_data = {
    "mode": "Connect",
    "message": msg,
    "status": sta
}

print(response_data)

Connected successfully
{'mode': 'Connect', 'message': 'Connected to COMCOM3!', 'status': 'success'}


# Read & Write as rroutine

In [49]:
a = 0
if not client.connect():
    print("Connection failed")
else:
    print("Connected")

    try:
        while True:
            # Read holding registers
            result = client.read_holding_registers(
                address=0,
                count=7,        # address 0, 1, 2, 3, 4, 5, 6
                slave=21
            )
            msg = {
                "mode": "STATS",
                "pos" : result.registers[0] if not result.isError() else None,
                "vel" : result.registers[1] if not result.isError() else None,
                "acc" : result.registers[2] if not result.isError() else None,
                "grip1" : result.registers[3] if not result.isError() else None,
                "grip2" : result.registers[4] if not result.isError() else None,
                "current" : result.registers[5] if not result.isError() else None,
                "emergency" : result.registers[6] if not result.isError() else None
            }
            print(msg)
            
            # if result.isError():
            #     print("Read error:", result)
            # else:
            #     print("Read value:", result.registers)

            # Write to a holding register
            result = client.write_register(
                address=2,
                value=a & 0xFFFF,   # Write the last 16 bits of a
                slave=21
            )

            a += 1
            time.sleep(0.2)   # 0.1 Hz
    except KeyboardInterrupt:
        print("Stopped by user")

    # client.close()

Connected
{'mode': 'STATS', 'pos': 1111, 'vel': 51916, 'acc': 67, 'grip1': 0, 'grip2': 0, 'current': 0, 'emergency': 0}
{'mode': 'STATS', 'pos': 1111, 'vel': 1794, 'acc': 0, 'grip1': 0, 'grip2': 0, 'current': 0, 'emergency': 0}
{'mode': 'STATS', 'pos': 1111, 'vel': 14502, 'acc': 1, 'grip1': 0, 'grip2': 0, 'current': 0, 'emergency': 0}
{'mode': 'STATS', 'pos': 1111, 'vel': 29378, 'acc': 2, 'grip1': 0, 'grip2': 0, 'current': 0, 'emergency': 0}
{'mode': 'STATS', 'pos': 1111, 'vel': 44306, 'acc': 3, 'grip1': 0, 'grip2': 0, 'current': 0, 'emergency': 0}
{'mode': 'STATS', 'pos': 1111, 'vel': 57321, 'acc': 4, 'grip1': 0, 'grip2': 0, 'current': 0, 'emergency': 0}
{'mode': 'STATS', 'pos': 1111, 'vel': 7775, 'acc': 5, 'grip1': 0, 'grip2': 0, 'current': 0, 'emergency': 0}
{'mode': 'STATS', 'pos': 1111, 'vel': 37564, 'acc': 6, 'grip1': 0, 'grip2': 0, 'current': 0, 'emergency': 0}
{'mode': 'STATS', 'pos': 1111, 'vel': 53621, 'acc': 7, 'grip1': 0, 'grip2': 0, 'current': 0, 'emergency': 0}
{'mode': '